# SPECTRON: free Colab LoRA training

Use only anonymized, expert-reviewed agricultural scenarios. This notebook trains an adapter; it does not make unreviewed advice safe for farmers.

In [ ]:
# In Colab: Runtime > Change runtime type > GPU, then run this cell.
import torch
assert torch.cuda.is_available(), 'No GPU is attached. Select a GPU runtime and reconnect.'
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Upload a ZIP containing software/local-ai-advisor. Do not include .venv, models, or private data.
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(zip_names) == 1, 'Upload exactly one local-ai-advisor ZIP file.'
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall('/content')
candidates = list(Path('/content').rglob('training/train_lora.py'))
assert len(candidates) == 1, 'Could not find training/train_lora.py inside the uploaded ZIP.'
PROJECT = candidates[0].parent.parent
print('Project:', PROJECT)

In [ ]:
# Upload the anonymized, expert-reviewed training data.
uploaded = files.upload()
data_names = [name for name in uploaded if name.endswith('.jsonl')]
assert len(data_names) == 1, 'Upload exactly one expert_scenarios.jsonl file.'
target = PROJECT / 'data' / 'expert_scenarios.jsonl'
target.write_bytes(Path(data_names[0]).read_bytes())
print('Training scenarios:', sum(1 for line in target.read_text(encoding='utf-8').splitlines() if line.strip()))

In [ ]:
# Install the project and training dependencies.
%cd {PROJECT}
!python -m pip uninstall -y torchao  # Colab may ship an incompatible optional torchao build; SPECTRON does not use it.
!python -m pip install --upgrade pip
!python -m pip install -e '.[training]'

In [ ]:
# Build a group-safe training/validation split.
!python training/build_dataset.py --source data/expert_scenarios.jsonl --output data/generated --seed 42
!wc -l data/generated/train.jsonl data/generated/validation.jsonl

In [ ]:
# Start LoRA training. On a free GPU, begin with 3 epochs and preserve the output.
!python training/train_lora.py --train data/generated/train.jsonl --validation data/generated/validation.jsonl --output models/qwen-agriassist-lora --epochs 3

In [ ]:
# Package and download the adapter immediately; a free Colab runtime can end at any time.
import shutil
archive = shutil.make_archive('/content/spectron-qwen-lora-adapter', 'zip', PROJECT / 'models' / 'qwen-agriassist-lora')
files.download(archive)

## Required next work

Download the adapter, test it against a separate held-out expert-reviewed set, then train the confidence calibrator. Do not connect it to farmer-facing recommendations until it passes agricultural safety review.